# 01 Data Collection - World GDP Data

Notebook ini untuk mengambil data GDP global dari World Bank API.

## Indikator yang diambil:
- GDP (NY.GDP.MKTP.CD)
- GDP Per Capita (NY.GDP.PCAP.CD)
- GNI (NY.GNP.ATLS.CD)
- Life Expectancy (SP.DYN.LE00.IN)
- Population (SP.POP.TOTL)
- Exports (NE.EXP.GNFS.CD)
- Imports (NE.IMP.GNFS.CD)

In [3]:
!pip install wbgapi

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [1]:
import pandas as pd
import wbgapi as wb
import os
from datetime import datetime

ModuleNotFoundError: No module named 'wbgapi'

In [ ]:
# Buat folder data jika belum ada
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

## Definisi Indikator World Bank

In [ ]:
# Mapping indikator World Bank
INDICATORS = {
    'GDP': 'NY.GDP.MKTP.CD',
    'GDP_Per_Capita': 'NY.GDP.PCAP.CD',
    'GNI': 'NY.GNP.ATLS.CD',
    'Life_Expectancy': 'SP.DYN.LE00.IN',
    'Population': 'SP.POP.TOTL',
    'Exports': 'NE.EXP.GNFS.CD',
    'Imports': 'NE.IMP.GNFS.CD'
}

# Rentang tahun
START_YEAR = 1960
END_YEAR = 2023

## Fungsi untuk mengambil data dari World Bank API

In [ ]:
def fetch_indicator_all_countries(indicator_code, start_year, end_year):
    """
    Mengambil data untuk satu indikator dari World Bank API untuk semua negara.
    Returns DataFrame dengan kolom: Country, Year, Value
    """
    try:
        data = wb.data.DataFrame(indicator_code, 
                                  economy='all', 
                                  time=range(start_year, end_year + 1), 
                                  labels=True,
                                  skipBlanks=True,
                                  columns='series')
        
        # Reset index
        data = data.reset_index()
        
        # Rename columns
        if len(data.columns) >= 2:
            data.columns = ['Country', 'Year', 'Value']
            # Extract year from Year column
            data['Year'] = data['Year'].str.extract(r'(\d+)').astype(float).astype('Int64')
            data['Value'] = pd.to_numeric(data['Value'], errors='coerce')
        
        return data
    except Exception as e:
        print(f"Error fetching {indicator_code}: {e}")
        return pd.DataFrame()

print("Fungsi fetch_indicator_all_countries berhasil dibuat")

## Ambil data untuk semua indikator

In [ ]:
# Dictionary untuk menyimpan DataFrame setiap indikator
indicator_dfs = {}

for name, code in INDICATORS.items():
    print(f"Mengambil data untuk {name} ({code})...")
    df = fetch_indicator_all_countries(code, START_YEAR, END_YEAR)
    
    if not df.empty:
        indicator_dfs[name] = df
        # Simpan individual file
        file_path = f'../data/raw/{name}.csv'
        df.to_csv(file_path, index=False)
        print(f"  Disimpan ke {file_path}")
        print(f"  Jumlah data: {len(df)} baris\n")
    else:
        print(f"  Gagal mengambil data untuk {name}\n")

## Gabungkan semua indikator menjadi satu dataset

In [ ]:
if len(indicator_dfs) > 0:
    # Mulai dengan GDP sebagai base
    if 'GDP' in indicator_dfs:
        dataset = indicator_dfs['GDP'].copy()
        dataset = dataset.rename(columns={'Value': 'GDP'})
        
        # Join dengan indikator lain
        for name, df in indicator_dfs.items():
            if name != 'GDP':
                dataset = dataset.merge(df.rename(columns={'Value': name}), 
                                        on=['Country', 'Year'], how='outer')
        
        # Sort by country and year
        dataset = dataset.sort_values(['Country', 'Year']).reset_index(drop=True)
        
        print("Dataset gabungan:")
        print(dataset.head(10))
        print(f"\nShape: {dataset.shape}")
        print(f"\nJumlah negara: {dataset['Country'].nunique()}")
        print(f"Rentang tahun: {dataset['Year'].min()} - {dataset['Year'].max()}")
    else:
        print("GDP data tidak tersedia")
else:
    print("Tidak ada data yang berhasil diambil")

## Filter Negara (Hapus agregat regional)

In [ ]:
# Daftar negara agregat yang akan dihapus
excluded_countries = [
    "Africa Eastern and Southern", "Africa Western and Central",
    "Arab World", "Caribbean small states", "Central Europe and the Baltics",
    "Early-demographic dividend", "East Asia & Pacific",
    "East Asia & Pacific (excluding high income)",
    "East Asia & Pacific (IDA & IBRD countries)",
    "Euro area", "Europe & Central Asia",
    "Europe & Central Asia (excluding high income)",
    "Europe & Central Asia (IDA & IBRD countries)",
    "European Union", "Fragile and conflict affected situations",
    "Heavily indebted poor countries (HIPC)", "High income",
    "IBRD only", "IDA & IBRD total", "IDA blend", "IDA only", "IDA total",
    "Late-demographic dividend", "Latin America & Caribbean",
    "Latin America & Caribbean (excluding high income)",
    "Latin America & the Caribbean (IDA & IBRD countries)",
    "Least developed countries: UN classification",
    "Low & middle income", "Low income", "Lower middle income",
    "Middle East & North Africa",
    "Middle East & North Africa (excluding high income)",
    "Middle East & North Africa (IDA & IBRD countries)",
    "Middle income", "North America", "Not classified",
    "OECD members", "Other small states",
    "Pacific island small states", "Pre-demographic dividend",
    "Small states", "South Asia", "South Asia (IDA & IBRD)",
    "Sub-Saharan Africa", "Sub-Saharan Africa (excluding high income)",
    "Sub-Saharan Africa (IDA & IBRD countries)",
    "Upper middle income", "World", "Post-demographic dividend"
]

if len(dataset) > 0:
    dataset_filtered = dataset[~dataset['Country'].isin(excluded_countries)].copy()
    print(f"Sebelum filter: {len(dataset)} baris, {dataset['Country'].nunique()} negara")
    print(f"Sesudah filter: {len(dataset_filtered)} baris, {dataset_filtered['Country'].nunique()} negara")
else:
    dataset_filtered = dataset

## Hitung Turunan dan Fitur Tambahan

In [ ]:
if len(dataset_filtered) > 0:
    # Sort by country and year
    dataset_filtered = dataset_filtered.sort_values(['Country', 'Year']).copy()
    
    # Hitung GDP_Growth (persentase change dari tahun ke tahun)
    dataset_filtered['prev_gdp'] = dataset_filtered.groupby('Country')['GDP'].shift(1)
    dataset_filtered['GDP_Growth'] = ((dataset_filtered['GDP'] - dataset_filtered['prev_gdp']) / dataset_filtered['prev_gdp'] * 100).round(2)
    
    # Hitung GDP dalam miliar USD
    dataset_filtered['GDP_B'] = (dataset_filtered['GDP'] / 1e9).round(2)
    
    # Hitung Exports dan Imports sebagai % GDP
    dataset_filtered['Exports_pct'] = (dataset_filtered['Exports'] / dataset_filtered['GDP'] * 100).round(2)
    dataset_filtered['Imports_pct'] = (dataset_filtered['Imports'] / dataset_filtered['GDP'] * 100).round(2)
    
    # Hitung Population Growth
    dataset_filtered['prev_pop'] = dataset_filtered.groupby('Country')['Population'].shift(1)
    dataset_filtered['Population_Growth'] = ((dataset_filtered['Population'] - dataset_filtered['prev_pop']) / dataset_filtered['prev_pop'] * 100).round(2)
    
    # Hitung lag features untuk GDP Growth
    dataset_filtered['GDP_Growth_lag1'] = dataset_filtered.groupby('Country')['GDP_Growth'].shift(1)
    dataset_filtered['GDP_Growth_lag2'] = dataset_filtered.groupby('Country')['GDP_Growth'].shift(2)
    dataset_filtered['GDP_lag1'] = dataset_filtered.groupby('Country')['GDP_B'].shift(1)
    
    print("Fitur tambahan berhasil dihitung")
    print(dataset_filtered.head())

## Simpan Dataset Gabungan

In [ ]:
if len(dataset_filtered) > 0:
    # Pilih kolom yang relevan
    final_cols = ['Country', 'Year', 'Exports', 'Imports', 'GDP', 'GDP_Per_Capita', 
                  'GNI', 'Life_Expectancy', 'Population', 'prev_gdp', 'GDP_Growth',
                  'GDP_B', 'Exports_pct', 'Imports_pct', 'Population_Growth',
                  'GDP_Growth_lag1', 'GDP_Growth_lag2', 'GDP_lag1']
    
    # Pastikan semua kolom ada
    available_cols = [col for col in final_cols if col in dataset_filtered.columns]
    dataset_final = dataset_filtered[available_cols].copy()
    
    # Simpan dataset mentah
    dataset_final.to_csv('../data/processed/dataset_world.csv', index=False)
    print("Dataset disimpan ke ../data/processed/dataset_world.csv")
    
    # Simpan info metadata
    metadata = {
        'start_year': START_YEAR,
        'end_year': END_YEAR,
        'indicators': INDICATORS,
        'total_rows': len(dataset_final),
        'total_countries': dataset_final['Country'].nunique(),
        'download_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    import json
    with open('../data/processed/metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print("Metadata disimpan ke ../data/processed/metadata.json")
else:
    print("Dataset kosong, tidak bisa disimpan")

## Ringkasan

In [ ]:
if len(dataset_final) > 0:
    print("="*50)
    print("DATA COLLECTION SELESAI")
    print("="*50)
    print(f"Rentang tahun: {START_YEAR} - {END_YEAR}")
    print(f"Jumlah negara: {metadata['total_countries']}")
    print(f"Total baris data: {metadata['total_rows']}")
    print(f"Jumlah indikator: {len(metadata['indicators'])}")
    print("\nFile yang dibuat:")
    print("- ../data/raw/[indikator].csv (file individual per indikator)")
    print("- ../data/processed/dataset_world.csv (dataset gabungan)")
    print("- ../data/processed/metadata.json (metadata)")
    
    print("\nFitur yang tersedia:")
    print(dataset_final.columns.tolist())
else:
    print("Data collection gagal")